In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class OxidizeAlcohol(MorphingOperator):
    def __init__(self):
        super(OxidizeAlcohol, self).__init__()
        self._name = "Oxidize Alcohol"
        self._target_atoms = [] 
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6X4;H1,H2]")

    def setOriginal(self, mol):
        super(OxidizeAlcohol, self).setOriginal(mol)
        self._target_atoms = []
        
        if not self.original:
            return
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return
            
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
        
            self._target_atoms.append((match[0], match[1]))

    def morph(self):
        if not self.original: 
            return None
            
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: 
            return None
            
        if not self._target_atoms:
            return MolpherMol(other=rdkit_mol)
            
        idx_o, idx_c = random.choice(self._target_atoms)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            if rw_mol.GetBondBetweenAtoms(idx_o, idx_c):
                rw_mol.RemoveBond(idx_o, idx_c)
            rw_mol.AddBond(idx_o, idx_c, Chem.BondType.DOUBLE)
            
            new_mol = rw_mol.GetMol()
    
            for idx in [idx_o, idx_c]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except Exception as e:
            return MolpherMol(other=rdkit_mol)

    def getName(self):
        return self._name

oxidize_alcohol_op = OxidizeAlcohol()
test_oxidize_molecules = {
    "1. 1-Προπανόλη (Πρωτοταγής Αλκοόλη)": "CCCO",
    "2. 2-Προπανόλη (Δευτεροταγής Αλκοόλη)": "CC(C)O",
    "3. tert-Βουτανόλη (Τριτοταγής - Απαγορευμένη)": "CC(C)(C)O",
    "4. Οξικό οξύ (Καρβοξύλιο - Κίνδυνος Valence Error)": "CC(=O)O"
}

print("=== STARTING OXIDIZE ALCOHOL TESTING ===")
for name, smiles in test_oxidize_molecules.items():
    test_mol = Chem.MolFromSmiles(smiles)
    if test_mol is None:
        continue
        
    mol = MolpherMol(smiles)
    oxidize_alcohol_op.setOriginal(mol)
    product = oxidize_alcohol_op.morph()
    
    print(f"\n{name}")
    print(f"  SOURCE: {mol.getSMILES()}")
    
    if product and product.getSMILES() != mol.getSMILES():
        print(f"  TARGET: {product.getSMILES()}")
    else:
        print("  TARGET: No change (Safe - Ignored)")
print("\n========================================")

=== STARTING OXIDIZE ALCOHOL TESTING ===

1. 1-Προπανόλη (Πρωτοταγής Αλκοόλη)
  SOURCE: CCCO
  TARGET: CCC=O

2. 2-Προπανόλη (Δευτεροταγής Αλκοόλη)
  SOURCE: CC(C)O
  TARGET: CC(C)=O

3. tert-Βουτανόλη (Τριτοταγής - Απαγορευμένη)
  SOURCE: CC(C)(C)O
  TARGET: No change (Safe - Ignored)

4. Οξικό οξύ (Καρβοξύλιο - Κίνδυνος Valence Error)
  SOURCE: CC(=O)O
  TARGET: No change (Safe - Ignored)

